# 04 · New ideas

Three ideas that the earlier rounds never tested, each aimed at information the base models cannot see:

| Idea | Uses labels? | Where it is computed |
| --- | --- | --- |
| **A. Population statistics** over train + test (frequencies, group-relative amounts, value popularity, repeat counts) | no | notebook 02 (`plus` columns) |
| **B. Repeat-applicant history**: default labels of *other* applications by the same probable person | yes | inside each CV fold |
| **C. Neighbour default rate**: default rate of the most similar applicants in external scores and loan term | yes | inside each CV fold |

**Leakage control for B and C.** Inside a fold, only labels of the fold's training part are used.
A training row never sees its own label: its features come from the *other* labelled rows
(its group-mates, its neighbours excluding itself). Validation and test rows use all training-part rows.
So every row gets "labels of other applicants", the same thing the test rows get.

**Decision rule (set before running).** Ideas are screened with LightGBM (fast) on two seeds against a control
using the same folds and settings. An idea is kept if it improves OOF AUC by at least **0.0002**. Kept ideas are
then confirmed on CatBoost, and only confirmed improvements go to the final blend (notebook 05).

In [1]:
import sys
sys.path.insert(0, '..')
from functools import lru_cache
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from src.utils import load_features, auc
from src.cv import run_cv, folds, median_iterations
from src import models

train, test, sets = load_features()
y = train.TARGET.to_numpy().astype(int)
BASE, RECON, PLUS = sets['base'], sets['base'] + sets['recon'], sets['plus']
LGB_PARAMS = dict(num_leaves=21, min_child_samples=89, colsample_bytree=0.685, subsample=0.962, subsample_freq=1,
                  reg_lambda=2.13, reg_alpha=3.69)
CAT_PARAMS = dict(depth=5, l2_leaf_reg=8, random_strength=0.5)
SCREEN_SEEDS = (0, 1)

## Label-based features (B and C)
**B.** For each row: number of other labelled applications with the same `APPLICANT_KEY`, how many of them
defaulted, and the rate.

**C.** Similarity space: the three external scores (missing values replaced by their reconstruction from
notebook 02) and the loan term, each standardised. For each row, the default rate among its 100 and 500
nearest training-part applicants.

In [2]:
def repeat_features(keys_lab, y_lab, keys_query, query_is_labelled):
    stats = pd.DataFrame({'key': keys_lab, 'y': y_lab}).groupby('key').y.agg(['size', 'sum'])
    n = keys_query.map(stats['size']).fillna(0).to_numpy()
    s = keys_query.map(stats['sum']).fillna(0).to_numpy()
    if query_is_labelled is not None:              # remove the row's own label
        n, s = n - 1, s - query_is_labelled
    return pd.DataFrame({'REPEAT_LABELLED': n, 'REPEAT_DEFAULTS': s, 'REPEAT_RATE': np.where(n > 0, s / np.maximum(n, 1), np.nan)})


def knn_space(df):
    cols = []
    for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
        cols.append(df[c].fillna(df['RECON_' + c]))
    cols.append(df.CREDIT_TERM.fillna(df.CREDIT_TERM.median()))
    return np.column_stack(cols)


SPACE_ALL = knn_space(pd.concat([train, test]))
SCALE = SPACE_ALL.std(axis=0)
SPACE_TRAIN, SPACE_TEST = knn_space(train) / SCALE, knn_space(test) / SCALE
KS = (100, 500)


def knn_features(index_points, y_index, query, exclude_self):
    k = max(KS) + int(exclude_self)
    finder = NearestNeighbors(n_neighbors=k, algorithm='kd_tree', n_jobs=8).fit(index_points)
    out = np.zeros((len(query), len(KS)))
    for start in range(0, len(query), 20000):
        ind = finder.kneighbors(query[start:start + 20000], return_distance=False)
        if exclude_self:                           # drop the row itself (query rows = index rows here)
            own = np.arange(start, start + len(ind))[:, None]
            keep = ind != own
            keep[~(~keep).any(axis=1), -1] = False  # self not among neighbours (exact duplicates): drop the farthest
            ind = ind[keep].reshape(len(ind), max(KS))
        labels = y_index[ind]
        for j, kk in enumerate(KS):
            out[start:start + len(ind), j] = labels[:, :kk].mean(axis=1)
    return pd.DataFrame(out, columns=[f'NEIGHBOUR_RATE_{kk}' for kk in KS])


@lru_cache(maxsize=None)
def label_features(seed, fold):
    tr, va = folds(y, seed)[fold]
    keys = train.APPLICANT_KEY
    parts = []
    for rows, query_keys, labelled, space, self_ in [(tr, keys.iloc[tr], y[tr], SPACE_TRAIN[tr], True),
                                                     (va, keys.iloc[va], None, SPACE_TRAIN[va], False),
                                                     (None, test.APPLICANT_KEY, None, SPACE_TEST, False)]:
        r = repeat_features(keys.iloc[tr], y[tr], query_keys, labelled)
        k = knn_features(SPACE_TRAIN[tr], y[tr], space, self_)
        parts.append(pd.concat([r, k], axis=1))
    return parts


def augmented(columns, extra):
    """CV augment hook: base columns + selected fold-dependent label features."""
    def augment(tr, va, seed, fold):
        f_tr, f_va, f_te = label_features(seed, fold)
        join = lambda X, F: pd.concat([X[columns].reset_index(drop=True), F[extra]], axis=1)
        return join(train.iloc[tr], f_tr), join(train.iloc[va], f_va), join(test, f_te)
    return augment


REPEAT = ['REPEAT_LABELLED', 'REPEAT_DEFAULTS', 'REPEAT_RATE']
NEIGHBOUR = [f'NEIGHBOUR_RATE_{k}' for k in KS]

Quick look before modelling: do the new label features separate defaulters on their own (seed 0, fold 0)?

In [3]:
tr, va = folds(y, 0)[0]
f_tr, f_va, _ = label_features(0, 0)
has_history = f_va.REPEAT_LABELLED.to_numpy() > 0
print(f'validation rows with a labelled repeat application: {has_history.mean():.2%}')
print('default rate | a previous application defaulted:',
      round(y[va][has_history & (f_va.REPEAT_DEFAULTS.to_numpy() > 0)].mean(), 3),
      '| none defaulted:', round(y[va][has_history & (f_va.REPEAT_DEFAULTS.to_numpy() == 0)].mean(), 3),
      '| overall:', round(y.mean(), 3))
for c in NEIGHBOUR:
    print(c, 'AUC on its own:', round(auc(y[va], f_va[c]), 4))
print('EXT_MEAN AUC on its own:', round(auc(y[va], -train.EXT_MEAN.iloc[va].fillna(0.5)), 4))

validation rows with a labelled repeat application: 2.69%
default rate | a previous application defaulted: 0.254 | none defaulted: 0.052 | overall: 0.081
NEIGHBOUR_RATE_100 AUC on its own: 0.7158
NEIGHBOUR_RATE_500 AUC on its own: 0.7243
EXT_MEAN AUC on its own: 0.7031


## Screening with LightGBM (development runs, seeds 0 and 1)

In [4]:
def lgb_run(name, columns, extra=()):
    extra = list(extra)
    _, _, info = run_cv(name, models.lightgbm(LGB_PARAMS), train[columns], y, test[columns], seeds=SCREEN_SEEDS,
                        augment=augmented(columns, extra) if extra else None, config=dict(params=LGB_PARAMS, extra=extra))
    return info


screen = {
    'control (base)': lgb_run('lgb_base_dev', BASE),
    'A  population statistics': lgb_run('lgb_plus_dev', BASE + PLUS),
    'B  repeat-applicant history': lgb_run('lgb_repeat_dev', BASE, REPEAT),
    'C  neighbour default rate': lgb_run('lgb_neighbour_dev', BASE, NEIGHBOUR),
    'A+B+C': lgb_run('lgb_abc_dev', BASE + PLUS, REPEAT + NEIGHBOUR),
}
control = screen['control (base)']['auc']
table = pd.DataFrame({name: dict(auc=i['auc'], gain=i['auc'] - control, seed0=i['seed_auc'][0], seed1=i['seed_auc'][1])
                      for name, i in screen.items()}).T
table.round(5)

  lgb_base_dev seed 0 fold 0: AUC 0.76058  {'iterations': 609}  (6s)


  lgb_base_dev seed 0 fold 1: AUC 0.75269  {'iterations': 590}  (6s)


  lgb_base_dev seed 0 fold 2: AUC 0.76145  {'iterations': 451}  (5s)


  lgb_base_dev seed 0 fold 3: AUC 0.76651  {'iterations': 533}  (5s)


  lgb_base_dev seed 0 fold 4: AUC 0.76659  {'iterations': 458}  (5s)


lgb_base_dev: OOF AUC 0.76288  per seed [0.7615, 0.76176]  (1.3 min of fitting)
lgb_plus_dev: OOF AUC 0.76272  per seed [0.7615, 0.76148]  (1.9 min of fitting)
lgb_repeat_dev: OOF AUC 0.76423  per seed [0.76261, 0.763]  (5.0 min of fitting)


lgb_neighbour_dev: OOF AUC 0.76294  per seed [0.76164, 0.7613]  (1.6 min of fitting)


lgb_abc_dev: OOF AUC 0.76400  per seed [0.76252, 0.76222]  (2.1 min of fitting)


,auc,gain,seed0,seed1
control (base),0.76288,0.00000,0.76150,0.76176
A population statistics,0.76272,-0.00016,0.76150,0.76148
B repeat-applicant history,0.76423,0.00135,0.76261,0.76300
C neighbour default rate,0.76294,0.00006,0.76164,0.76130
A+B+C,0.76400,0.00112,0.76252,0.76222


Per-fold gains show whether an improvement is consistent or driven by one fold.

In [5]:
def fold_gain(info):
    a = pd.DataFrame(info['folds']).set_index(['seed', 'fold']).auc
    b = pd.DataFrame(screen['control (base)']['folds']).set_index(['seed', 'fold']).auc
    return (a - b)

pd.DataFrame({name: fold_gain(i) for name, i in screen.items() if name != 'control (base)'}).round(5)

A  population statistics  B  repeat-applicant history  \
seed fold                                                          
0    0                     -0.00012                      0.00009   
     1                      0.00012                      0.00098   
     2                      0.00054                      0.00070   
     3                     -0.00074                      0.00113   
     4                     -0.00001                      0.00265   
1    0                     -0.00107                      0.00205   
     1                      0.00042                      0.00135   
     2                     -0.00042                      0.00153   
     3                     -0.00013                      0.00172   
     4                     -0.00033                     -0.00044   

           C  neighbour default rate    A+B+C  
seed fold                                      
0    0                       0.00021  0.00150  
     1                       0.00154  0.00137  
     2                      -0.00094  0.00046  
     3                      -0.00068 -0.00085  
     4                       0.00054  0.00264  
1    0                      -0.00055  0.00051  
     1                      -0.00118 -0.00013  
     2                      -0.00133 -0.00003  
     3                       0.00029  0.00084  
     4                       0.00045  0.00094

## Decision and confirmation on CatBoost
Ideas passing the 0.0002 rule are combined; the combination is then run with CatBoost-recon
(the strongest family) against the CatBoost-recon development run from notebook 03.

In [6]:
kept = [name for name in ['A  population statistics', 'B  repeat-applicant history', 'C  neighbour default rate']
        if table.loc[name, 'gain'] >= 0.0002]
print('kept ideas:', kept)
KEEP_COLUMNS = PLUS if 'A  population statistics' in kept else []
KEEP_EXTRA = (REPEAT if 'B  repeat-applicant history' in kept else []) + (NEIGHBOUR if 'C  neighbour default rate' in kept else [])

kept ideas: ['B  repeat-applicant history']


In [7]:
if kept:
    lgb_kept = lgb_run('lgb_kept_dev', BASE + KEEP_COLUMNS, KEEP_EXTRA)
    print(f'LightGBM kept-ideas gain: {lgb_kept["auc"] - control:+.5f}')
    cols = RECON + KEEP_COLUMNS
    _, _, cat_new_dev = run_cv('cat_new_dev', models.catboost(CAT_PARAMS), train[cols], y, test[cols],
                               augment=augmented(cols, KEEP_EXTRA) if KEEP_EXTRA else None,
                               config=dict(params=CAT_PARAMS, extra=KEEP_EXTRA))
    _, _, cat_ref = run_cv('cat_recon_dev', models.catboost(CAT_PARAMS), train[RECON], y, test[RECON], config=dict(params=CAT_PARAMS))
    print(f'CatBoost-recon gain from kept ideas (seed 0): {cat_new_dev["auc"] - cat_ref["auc"]:+.5f}')

lgb_kept_dev: OOF AUC 0.76423  per seed [0.76261, 0.763]  (1.7 min of fitting)
LightGBM kept-ideas gain: +0.00135
cat_new_dev: OOF AUC 0.76439  per seed [0.76439]  (17.6 min of fitting)


cat_recon_dev: OOF AUC 0.76329  per seed [0.76329]  (8.0 min of fitting)
CatBoost-recon gain from kept ideas (seed 0): +0.00111


## Final runs of the improved families (fixed length, 3 seeds)
Same protocol as notebook 03. Whether these replace or join the base models is decided in notebook 05
by the blend.

In [8]:
SEEDS = (0, 1, 2)
if kept:
    n = median_iterations(lgb_kept)
    run_cv('lgb_new', models.lightgbm(LGB_PARAMS, iterations=n), train[BASE + KEEP_COLUMNS], y, test[BASE + KEEP_COLUMNS],
           seeds=SEEDS, augment=augmented(BASE + KEEP_COLUMNS, KEEP_EXTRA) if KEEP_EXTRA else None,
           config=dict(params=LGB_PARAMS, iterations=n, extra=KEEP_EXTRA))
    n = median_iterations(cat_new_dev)
    run_cv('cat_new', models.catboost(CAT_PARAMS, iterations=n), train[cols], y, test[cols], seeds=SEEDS,
           augment=augmented(cols, KEEP_EXTRA) if KEEP_EXTRA else None,
           config=dict(params=CAT_PARAMS, iterations=n, extra=KEEP_EXTRA))

lgb_new: OOF AUC 0.76438  per seed [0.76238, 0.76267, 0.76268]  (3.5 min of fitting)


cat_new: OOF AUC 0.76613  per seed [0.76422, 0.76485, 0.76456]  (30.4 min of fitting)


The improved tree models fully replaced the base tree models in a preliminary blend, so the neural network
gets the same kept features (same 40-epoch recipe as notebook 03).

In [9]:
if KEEP_EXTRA:
    run_cv('tabm_new', models.tabm(iterations=40), train[BASE + KEEP_COLUMNS], y, test[BASE + KEEP_COLUMNS], seeds=SEEDS,
           augment=augmented(BASE + KEEP_COLUMNS, KEEP_EXTRA), config=dict(epochs=40, extra=KEEP_EXTRA))

tabm_new: OOF AUC 0.76515  per seed [0.76185, 0.76031, 0.76137]  (373.9 min of fitting)
